# DATA 255 Lab 1 - Image Classification on NPU
**Spring 2026**

This notebook:
1. Mounts Google Drive and loads the ImageNet-20 dataset
2. Builds a custom CNN from scratch (unique architecture)
3. Trains with augmentation to achieve 60%+ accuracy
4. Exports the model to ONNX format for NPU deployment

## Step 0: Mount Google Drive & Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
import zipfile

# ── CONFIGURE YOUR DRIVE FOLDER ──
DRIVE_FOLDER = '/content/drive/MyDrive/DATA 255 Lab-1'
WORK_DIR     = '/content/data255_lab1'
os.makedirs(WORK_DIR, exist_ok=True)

# ── UNZIP DATASETS ────────────────────────────────────────────────────────────
def unzip_if_needed(zip_path, extract_to):
    if not os.path.exists(extract_to):
        print(f'Extracting {os.path.basename(zip_path)} ...')
        with zipfile.ZipFile(zip_path, 'r') as z:
            z.extractall(extract_to)
        print(f'  → done: {extract_to}')
    else:
        print(f'Already extracted: {extract_to}')

unzip_if_needed(f'{DRIVE_FOLDER}/imagenet_train20.zip', f'{WORK_DIR}/train')
unzip_if_needed(f'{DRIVE_FOLDER}/imagenet_val20.zip',   f'{WORK_DIR}/val')

TRAIN_DIR = f'{WORK_DIR}/train'
VAL_DIR   = f'{WORK_DIR}/val'
LABEL_TXT_TRAIN = f'{DRIVE_FOLDER}/imagenet_train20.txt'
LABEL_TXT_VAL   = f'{DRIVE_FOLDER}/imagenet_val20.txt'

print('Train dir:', TRAIN_DIR)
print('Val dir:  ', VAL_DIR)

In [ ]:
# Inspect the extracted structure so we know how to build the Dataset
import os

train_contents = os.listdir(TRAIN_DIR)
print('Train folder top-level entries (first 10):', sorted(train_contents)[:10])
print('Total entries in train:', len(train_contents))

val_contents = os.listdir(VAL_DIR)
print('Val folder top-level entries (first 10):', sorted(val_contents)[:10])
print('Total entries in val:', len(val_contents))

## Step 1: Dataset – Parse Label Files & Build PyTorch Dataset

The `.txt` files typically look like:
```
n01440764/ILSVRC2012_val_00000293.JPEG 0
n01695060/ILSVRC2012_val_00000088.JPEG 1
...
```
We parse these to map image paths → integer labels.

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
import numpy as np
import time
import copy

print('PyTorch version:', torch.__version__)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)

In [ ]:
class ImageNet20Dataset(Dataset):
    """
    Reads image paths + labels from the provided .txt file.
    Expected format per line:  <relative_path>  <label_int>
    e.g.   n01440764/img.JPEG 0
    """
    def __init__(self, root_dir, label_txt, transform=None):
        self.root_dir  = root_dir
        self.transform = transform
        self.samples   = []

        with open(label_txt, 'r') as f:
            for line in f:
                line = line.strip()
                if not line:
                    continue
                parts = line.split()
                # handle both "path label" and just filenames with folder=class
                if len(parts) >= 2:
                    rel_path = parts[0]
                    label    = int(parts[-1])
                else:
                    rel_path = parts[0]
                    label    = -1  # fallback – shouldn't happen
                full_path = os.path.join(root_dir, rel_path)
                if os.path.exists(full_path):
                    self.samples.append((full_path, label))

        print(f'  Loaded {len(self.samples)} samples from {label_txt}')

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path, label = self.samples[idx]
        img = Image.open(path).convert('RGB')
        if self.transform:
            img = self.transform(img)
        return img, label

In [ ]:
# ── IMAGE SIZE required by lab: 240×240 ───────────────────────────────────────
IMG_SIZE = 240

# ImageNet mean/std (standard for ImageNet-trained pipelines)
MEAN = [0.485, 0.456, 0.406]
STD  = [0.229, 0.224, 0.225]

train_transform = transforms.Compose([
    transforms.RandomResizedCrop(IMG_SIZE, scale=(0.6, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(p=0.1),
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.3, hue=0.1),
    transforms.RandomRotation(15),
    transforms.RandomGrayscale(p=0.05),
    transforms.ToTensor(),
    transforms.Normalize(mean=MEAN, std=STD),
    transforms.RandomErasing(p=0.2, scale=(0.02, 0.15))
])

val_transform = transforms.Compose([
    transforms.Resize(int(IMG_SIZE * 1.1)),
    transforms.CenterCrop(IMG_SIZE),
    transforms.ToTensor(),
    transforms.Normalize(mean=MEAN, std=STD),
])

print('Building datasets ...')
train_dataset = ImageNet20Dataset(TRAIN_DIR, LABEL_TXT_TRAIN, transform=train_transform)
val_dataset   = ImageNet20Dataset(VAL_DIR,   LABEL_TXT_VAL,   transform=val_transform)

BATCH_SIZE = 64

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=4, pin_memory=True, drop_last=True)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=4, pin_memory=True)

print(f'Train batches: {len(train_loader)} | Val batches: {len(val_loader)}')

## Step 2: Build Custom CNN Architecture from Scratch

**Architecture: DualPathNet-20** – a unique design combining:
- Inverted residual blocks (inspired by MobileNetV2 style but with different expansion ratios & layout)
- Squeeze-and-Excitation attention gates
- Dual-path feature merging
- Progressive channel widening

All custom – no pretrained weights.

In [ ]:
class SEBlock(nn.Module):
    """Squeeze-and-Excitation channel attention."""
    def __init__(self, channels, reduction=8):
        super().__init__()
        mid = max(8, channels // reduction)
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.fc   = nn.Sequential(
            nn.Linear(channels, mid, bias=False),
            nn.ReLU(inplace=True),
            nn.Linear(mid, channels, bias=False),
            nn.Sigmoid()
        )

    def forward(self, x):
        b, c, _, _ = x.size()
        w = self.pool(x).view(b, c)
        w = self.fc(w).view(b, c, 1, 1)
        return x * w


class DepthwiseSepConv(nn.Module):
    """Depthwise separable conv block with BN + ReLU."""
    def __init__(self, in_ch, out_ch, stride=1):
        super().__init__()
        self.dw = nn.Sequential(
            nn.Conv2d(in_ch, in_ch, 3, stride=stride, padding=1, groups=in_ch, bias=False),
            nn.BatchNorm2d(in_ch),
            nn.ReLU6(inplace=True)
        )
        self.pw = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU6(inplace=True)
        )

    def forward(self, x):
        return self.pw(self.dw(x))


class InvResBlock(nn.Module):
    """
    Inverted residual block:
      expand → depthwise → project → SE gate
    Residual connection only when stride==1 and in_ch==out_ch.
    """
    def __init__(self, in_ch, out_ch, stride=1, expand=4):
        super().__init__()
        mid = in_ch * expand
        self.use_skip = (stride == 1 and in_ch == out_ch)

        self.block = nn.Sequential(
            # Expand
            nn.Conv2d(in_ch, mid, 1, bias=False),
            nn.BatchNorm2d(mid),
            nn.ReLU6(inplace=True),
            # Depthwise
            nn.Conv2d(mid, mid, 3, stride=stride, padding=1, groups=mid, bias=False),
            nn.BatchNorm2d(mid),
            nn.ReLU6(inplace=True),
            # Project
            nn.Conv2d(mid, out_ch, 1, bias=False),
            nn.BatchNorm2d(out_ch),
        )
        self.se = SEBlock(out_ch)

    def forward(self, x):
        out = self.se(self.block(x))
        if self.use_skip:
            out = out + x
        return out


class DualPathNet20(nn.Module):
    """
    Custom DualPathNet-20 for ImageNet-20 classification.
    Input:  (B, 3, 240, 240)
    Output: (B, 20)

    Dual-path design: two parallel streams (wide & narrow) that
    merge their features at multiple stages before classification.
    """
    def __init__(self, num_classes=20, dropout=0.4):
        super().__init__()

        # ── Stem (shared) ─────────────────────────────────────────────────────
        self.stem = nn.Sequential(
            nn.Conv2d(3, 32, 3, stride=2, padding=1, bias=False),   # 120×120
            nn.BatchNorm2d(32),
            nn.ReLU6(inplace=True),
            nn.Conv2d(32, 48, 3, stride=1, padding=1, bias=False),  # 120×120
            nn.BatchNorm2d(48),
            nn.ReLU6(inplace=True),
        )

        # ── Path A (wider/deeper) ─────────────────────────────────────────────
        self.pathA = nn.Sequential(
            InvResBlock(48,  64,  stride=2, expand=4),   # 60×60
            InvResBlock(64,  64,  stride=1, expand=4),
            InvResBlock(64,  96,  stride=2, expand=6),   # 30×30
            InvResBlock(96,  96,  stride=1, expand=6),
            InvResBlock(96,  96,  stride=1, expand=6),
            InvResBlock(96,  128, stride=2, expand=6),   # 15×15
            InvResBlock(128, 128, stride=1, expand=6),
            InvResBlock(128, 160, stride=1, expand=6),
        )

        # ── Path B (narrower, different expand ratios) ────────────────────────
        self.pathB = nn.Sequential(
            DepthwiseSepConv(48,  64,  stride=2),        # 60×60
            InvResBlock(64,  80,  stride=2, expand=3),   # 30×30
            InvResBlock(80,  80,  stride=1, expand=3),
            InvResBlock(80,  112, stride=2, expand=4),   # 15×15
            InvResBlock(112, 112, stride=1, expand=4),
            InvResBlock(112, 160, stride=1, expand=4),
        )

        # ── Merge & Head ──────────────────────────────────────────────────────
        # After merge: 160 + 160 = 320 channels at 15×15
        self.merge_conv = nn.Sequential(
            nn.Conv2d(320, 256, 1, bias=False),
            nn.BatchNorm2d(256),
            nn.ReLU6(inplace=True),
            InvResBlock(256, 256, stride=2, expand=6),   # 8×8
            InvResBlock(256, 320, stride=1, expand=6),
            nn.Conv2d(320, 512, 1, bias=False),
            nn.BatchNorm2d(512),
            nn.ReLU6(inplace=True),
        )

        self.pool       = nn.AdaptiveAvgPool2d(1)
        self.classifier = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(512, 256),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout * 0.5),
            nn.Linear(256, num_classes)
        )

        # Weight initialisation
        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
                if m.bias is not None:
                    nn.init.zeros_(m.bias)
            elif isinstance(m, nn.BatchNorm2d):
                nn.init.ones_(m.weight)
                nn.init.zeros_(m.bias)
            elif isinstance(m, nn.Linear):
                nn.init.normal_(m.weight, 0, 0.01)
                if m.bias is not None:
                    nn.init.zeros_(m.bias)

    def forward(self, x):
        x = self.stem(x)
        a = self.pathA(x)
        b = self.pathB(x)
        x = torch.cat([a, b], dim=1)  # dual-path merge
        x = self.merge_conv(x)
        x = self.pool(x)
        x = x.view(x.size(0), -1)
        return self.classifier(x)


# Instantiate model
model = DualPathNet20(num_classes=20, dropout=0.35).to(device)

# Quick sanity check
with torch.no_grad():
    dummy = torch.randn(2, 3, 240, 240).to(device)
    out   = model(dummy)
print('Output shape:', out.shape)  # should be (2, 20)

total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Trainable parameters: {total_params:,}')

## Step 3: Training Loop with LR Scheduling + Label Smoothing

In [ ]:
# ── Hyperparameters ────────────────────────────────────────────────────────────
NUM_EPOCHS     = 30
LR             = 0.001
WEIGHT_DECAY   = 1e-4
LABEL_SMOOTH   = 0.1

criterion = nn.CrossEntropyLoss(label_smoothing=LABEL_SMOOTH)

optimizer = optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)

# Cosine annealing with warm restart
scheduler = optim.lr_scheduler.CosineAnnealingWarmRestarts(
    optimizer, T_0=10, T_mult=2, eta_min=1e-6
)

In [ ]:
def mixup_data(x, y, alpha=0.3):
    """Mixup augmentation."""
    if alpha > 0:
        lam = np.random.beta(alpha, alpha)
    else:
        lam = 1.0
    bsz  = x.size(0)
    idx  = torch.randperm(bsz).to(x.device)
    mixed_x  = lam * x + (1 - lam) * x[idx, :]
    y_a, y_b = y, y[idx]
    return mixed_x, y_a, y_b, lam


def mixup_criterion(criterion, pred, y_a, y_b, lam):
    return lam * criterion(pred, y_a) + (1 - lam) * criterion(pred, y_b)


def train_one_epoch(model, loader, optimizer, criterion, device, use_mixup=True):
    model.train()
    total_loss = 0.0
    correct    = 0
    total      = 0

    for imgs, labels in loader:
        imgs, labels = imgs.to(device), labels.to(device)

        if use_mixup and np.random.rand() > 0.5:
            imgs, y_a, y_b, lam = mixup_data(imgs, labels)
            outputs = model(imgs)
            loss    = mixup_criterion(criterion, outputs, y_a, y_b, lam)
        else:
            outputs = model(imgs)
            loss    = criterion(outputs, labels)

        optimizer.zero_grad()
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
        optimizer.step()

        total_loss += loss.item() * imgs.size(0)
        _, preds    = outputs.max(1)
        correct    += preds.eq(labels).sum().item()
        total      += labels.size(0)

    return total_loss / total, 100.0 * correct / total


@torch.no_grad()
def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss = 0.0
    correct    = 0
    total      = 0

    for imgs, labels in loader:
        imgs, labels = imgs.to(device), labels.to(device)
        outputs      = model(imgs)
        loss         = criterion(outputs, labels)

        total_loss += loss.item() * imgs.size(0)
        _, preds    = outputs.max(1)
        correct    += preds.eq(labels).sum().item()
        total      += labels.size(0)

    return total_loss / total, 100.0 * correct / total

In [ ]:
# ── Training loop ──────────────────────────────────────────────────────────────
MODEL_SAVE_PATH = f'{WORK_DIR}/best_model.pth'

best_val_acc  = 0.0
history       = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}

print(f'Starting training for {NUM_EPOCHS} epochs on {device} ...\n')
print(f'{"Epoch":>6} | {"Train Loss":>10} | {"Train Acc":>9} | {"Val Loss":>9} | {"Val Acc":>8} | {"LR":>10}')
print('-' * 70)

for epoch in range(1, NUM_EPOCHS + 1):
    t0 = time.time()

    # Disable mixup in later epochs for fine-tuning
    use_mixup = epoch <= int(NUM_EPOCHS * 0.8)

    tr_loss, tr_acc = train_one_epoch(model, train_loader, optimizer, criterion, device, use_mixup)
    vl_loss, vl_acc = evaluate(model, val_loader, criterion, device)

    scheduler.step()

    history['train_loss'].append(tr_loss)
    history['train_acc'].append(tr_acc)
    history['val_loss'].append(vl_loss)
    history['val_acc'].append(vl_acc)

    # Save best model
    if vl_acc > best_val_acc:
        best_val_acc = vl_acc
        torch.save(model.state_dict(), MODEL_SAVE_PATH)
        flag = ' ← best'
    else:
        flag = ''

    cur_lr = optimizer.param_groups[0]['lr']
    elapsed = time.time() - t0
    print(f'{epoch:>6} | {tr_loss:>10.4f} | {tr_acc:>8.2f}% | {vl_loss:>9.4f} | {vl_acc:>7.2f}%  | {cur_lr:>10.2e}  {elapsed:.0f}s{flag}')

print(f'\nTraining complete. Best Val Acc: {best_val_acc:.2f}%')

In [ ]:
# ── Plot training history ──────────────────────────────────────────────────────
import matplotlib.pyplot as plt

epochs_range = range(1, NUM_EPOCHS + 1)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(epochs_range, history['train_loss'], label='Train Loss')
axes[0].plot(epochs_range, history['val_loss'],   label='Val Loss')
axes[0].set_title('Loss'); axes[0].set_xlabel('Epoch'); axes[0].legend()

axes[1].plot(epochs_range, history['train_acc'], label='Train Acc')
axes[1].plot(epochs_range, history['val_acc'],   label='Val Acc')
axes[1].axhline(y=60, color='r', linestyle='--', label='60% target')
axes[1].set_title('Accuracy (%)'); axes[1].set_xlabel('Epoch'); axes[1].legend()

plt.tight_layout()
plt.savefig(f'{WORK_DIR}/training_history.png', dpi=120)
plt.show()
print(f'Best Validation Accuracy: {best_val_acc:.2f}%')

## Step 4: Load Best Model & Export to ONNX

In [ ]:
# Load best weights
model.load_state_dict(torch.load(MODEL_SAVE_PATH, map_location=device))
model.eval()
print('Best model loaded.')

# Final validation accuracy
_, final_acc = evaluate(model, val_loader, criterion, device)
print(f'Final val accuracy (best model): {final_acc:.2f}%')

In [ ]:
import torch.onnx

ONNX_PATH = f'{WORK_DIR}/model_imagenet20.onnx'

dummy_input = torch.randn(1, 3, 240, 240).to(device)

torch.onnx.export(
    model,
    dummy_input,
    ONNX_PATH,
    export_params=True,
    opset_version=11,
    do_constant_folding=True,
    input_names=['input'],
    output_names=['output'],
    dynamic_axes={'input': {0: 'batch_size'}, 'output': {0: 'batch_size'}}
)

size_mb = os.path.getsize(ONNX_PATH) / (1024 * 1024)
print(f'ONNX model saved: {ONNX_PATH}  ({size_mb:.1f} MB)')

In [ ]:
# ── Verify ONNX model ──────────────────────────────────────────────────────────
import onnx
import onnxruntime as ort

onnx_model = onnx.load(ONNX_PATH)
onnx.checker.check_model(onnx_model)
print('ONNX model is valid!')

# Run one inference with ONNX Runtime to confirm
sess   = ort.InferenceSession(ONNX_PATH)
inp    = dummy_input.cpu().numpy()
result = sess.run(None, {'input': inp})
print('ONNX inference output shape:', result[0].shape)

In [ ]:
# ── Quick ONNX accuracy check ──────────────────────────────────────────────────
# (useful to verify ONNX behaves same as PyTorch)
ort_session = ort.InferenceSession(ONNX_PATH)

correct = 0
total   = 0

for imgs, labels in val_loader:
    inp     = imgs.numpy()
    outputs = ort_session.run(None, {'input': inp})[0]
    preds   = np.argmax(outputs, axis=1)
    correct += (preds == labels.numpy()).sum()
    total   += labels.size(0)

onnx_acc = 100.0 * correct / total
print(f'ONNX Runtime Val Accuracy: {onnx_acc:.2f}%')

## Step 5: Prepare Calibration Images for MXQ Compilation

We'll copy a subset of training images to a `calibration_images` folder – the ONNX→MXQ script needs these.

In [ ]:
import shutil
import random

CALIB_DIR  = f'{WORK_DIR}/calibration_images'
NUM_CALIB  = 200  # ~10 per class

os.makedirs(CALIB_DIR, exist_ok=True)

# Collect all training image paths
all_train_paths = [s[0] for s in train_dataset.samples]
random.seed(42)
selected = random.sample(all_train_paths, min(NUM_CALIB, len(all_train_paths)))

for src in selected:
    dst = os.path.join(CALIB_DIR, os.path.basename(src))
    if not os.path.exists(dst):
        shutil.copy2(src, dst)

print(f'Copied {len(os.listdir(CALIB_DIR))} calibration images to {CALIB_DIR}')

## Step 6: Copy Files to Google Drive Output Folder

The 5 files needed in your Drive output folder for the NPU lab:
1. `best_model.pth`  
2. `model_imagenet20.onnx`  
3. `calibration_images.zip`  
4. `onnx__mxq_conversion.py`  
5. `npu_valid.py`  

In [ ]:
OUTPUT_DRIVE = f'{DRIVE_FOLDER}/lab1_outputs'
os.makedirs(OUTPUT_DRIVE, exist_ok=True)

# 1. best_model.pth
shutil.copy2(MODEL_SAVE_PATH, f'{OUTPUT_DRIVE}/best_model.pth')

# 2. model_imagenet20.onnx
shutil.copy2(ONNX_PATH, f'{OUTPUT_DRIVE}/model_imagenet20.onnx')

# 3. calibration_images.zip
calib_zip = f'{OUTPUT_DRIVE}/calibration_images.zip'
with zipfile.ZipFile(calib_zip, 'w', zipfile.ZIP_DEFLATED) as zf:
    for fn in os.listdir(CALIB_DIR):
        zf.write(os.path.join(CALIB_DIR, fn), fn)
print(f'calibration_images.zip saved ({os.path.getsize(calib_zip)/1e6:.1f} MB)')

print('Files copied to Drive:', os.listdir(OUTPUT_DRIVE))

In [ ]:
# 4. Write onnx__mxq_conversion.py  (customized for your 240×240 model)
conversion_script = '''
import os
import numpy as np
from PIL import Image
import shutil
import zipfile
from qubee import mxq_compile
from qubee.calibration import make_calib_man

# ── Config ────────────────────────────────────────────────────────────────────
ONNX_FILE    = "model_imagenet20.onnx"
MXQ_FILE     = "model_imagenet20.mxq"
CALIB_ZIP    = "calibration_images.zip"
CALIB_IMAGES = "calibration_images"
CALIB_DATA   = "calibration_data"

# Unzip calibration images if needed
if not os.path.exists(CALIB_IMAGES):
    print("Extracting calibration images ...")
    with zipfile.ZipFile(CALIB_ZIP, "r") as z:
        z.extractall(CALIB_IMAGES)

# ── Preprocessing: match exactly what training used ───────────────────────────
def preprocess_imagenet(image_path):
    img = Image.open(image_path).convert("RGB")
    # Resize + center-crop to 240×240
    img = img.resize((int(240 * 1.1), int(240 * 1.1)), Image.BILINEAR)
    w, h = img.size
    left   = (w - 240) // 2
    top    = (h - 240) // 2
    img    = img.crop((left, top, left + 240, top + 240))
    arr    = np.array(img, dtype=np.float32) / 255.0
    mean   = np.array([0.485, 0.456, 0.406])
    std    = np.array([0.229, 0.224, 0.225])
    arr    = (arr - mean) / std
    # HWC → CHW  (Qubee expects CHW for ONNX)
    arr    = arr.transpose(2, 0, 1)
    return arr.astype(np.float32)

# ── Build calibration dataset ─────────────────────────────────────────────────
if os.path.exists(CALIB_DATA):
    shutil.rmtree(CALIB_DATA)

num_calib = len(os.listdir(CALIB_IMAGES))
print(f"Building calibration data from {num_calib} images ...")

make_calib_man(
    pre_ftn   = preprocess_imagenet,
    data_dir  = CALIB_IMAGES,
    save_dir  = ".",
    save_name = CALIB_DATA,
    max_size  = num_calib
)

# ── Compile ONNX → MXQ ────────────────────────────────────────────────────────
print("Compiling ONNX → MXQ ...")
mxq_compile(
    model              = ONNX_FILE,
    calib_data_path    = CALIB_DATA,
    quantize_method    = "maxpercentile",
    is_quant_ch        = True,
    quantize_percentile= 0.999,
    topk_ratio         = 0.01,
    quant_output       = "layer",
    save_path          = MXQ_FILE,
    backend            = "onnx"
)

print(f"Compiled: {MXQ_FILE} ({os.path.getsize(MXQ_FILE)/(1024*1024):.1f} MB)")
'''

conv_path = f'{OUTPUT_DRIVE}/onnx__mxq_conversion.py'
with open(conv_path, 'w') as f:
    f.write(conversion_script.strip())
print('Wrote:', conv_path)

In [ ]:
# 5. Write npu_valid.py  (validation script for NPU)
npu_script = '''
import os
import numpy as np
from PIL import Image
import maccel

# ── Config ────────────────────────────────────────────────────────────────────
MXQ_FILE   = "model_imagenet20.mxq"
VAL_DIR    = "imagenet_val20"          # unzipped validation folder
LABEL_TXT  = "imagenet_val20.txt"      # path  label  per line
IMG_SIZE   = 240

# ── Preprocessing (must match training) ──────────────────────────────────────
MEAN = np.array([0.485, 0.456, 0.406], dtype=np.float32)
STD  = np.array([0.229, 0.224, 0.225], dtype=np.float32)

def preprocess(image_path):
    img  = Image.open(image_path).convert("RGB")
    img  = img.resize((int(IMG_SIZE * 1.1), int(IMG_SIZE * 1.1)), Image.BILINEAR)
    w, h = img.size
    left = (w - IMG_SIZE) // 2
    top  = (h - IMG_SIZE) // 2
    img  = img.crop((left, top, left + IMG_SIZE, top + IMG_SIZE))
    arr  = np.array(img, dtype=np.float32) / 255.0
    arr  = (arr - MEAN) / STD
    arr  = arr.transpose(2, 0, 1)                      # HWC → CHW
    arr  = np.expand_dims(arr, 0).astype(np.float32)   # → (1, 3, 240, 240)
    return arr

# ── Load label file ───────────────────────────────────────────────────────────
samples = []
with open(LABEL_TXT) as f:
    for line in f:
        parts = line.strip().split()
        if len(parts) >= 2:
            rel_path = parts[0]
            label    = int(parts[-1])
            full_path = os.path.join(VAL_DIR, rel_path)
            if os.path.exists(full_path):
                samples.append((full_path, label))

print(f"Loaded {len(samples)} validation samples.")

# ── Run NPU inference ─────────────────────────────────────────────────────────
device    = maccel.Accelerator(0)
npu_model = maccel.Model(MXQ_FILE)
npu_model.launch(device)

correct = 0
total   = 0

for path, label in samples:
    inp     = preprocess(path)
    outputs = npu_model.infer([inp])
    pred    = int(np.argmax(outputs[0]))
    if pred == label:
        correct += 1
    total += 1
    if total % 100 == 0:
        print(f"  [{total}/{len(samples)}]  Running acc: {100*correct/total:.2f}%")

npu_model.dispose()
print(f"\nFinal NPU Accuracy: {100*correct/total:.2f}%  ({correct}/{total})")
'''

npu_path = f'{OUTPUT_DRIVE}/npu_valid.py'
with open(npu_path, 'w') as f:
    f.write(npu_script.strip())
print('Wrote:', npu_path)

print('\n=== All 5 output files in Drive ===')
for fn in sorted(os.listdir(OUTPUT_DRIVE)):
    sz = os.path.getsize(f'{OUTPUT_DRIVE}/{fn}')
    print(f'  {fn:40s}  {sz/1024:.1f} KB')

## Summary

| File | Purpose |
|------|---------|
| `best_model.pth` | PyTorch weights (backup) |
| `model_imagenet20.onnx` | ONNX model for NPU compilation |
| `calibration_images.zip` | Calibration data for quantization |
| `onnx__mxq_conversion.py` | Run inside Docker to produce `.mxq` |
| `npu_valid.py` | Run on Windows (outside Docker) on NPU |

### NPU Lab Steps (from Steps_for_lab.pdf)
1. Copy the 5 files to `C:\NPU_Projects`  
2. Launch Docker and run `onnx__mxq_conversion.py` → produces `model_imagenet20.mxq`  
3. Exit Docker, run `npu_valid.py` on Windows to measure NPU accuracy  